In [5]:
from scipy import linalg
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize._linesearch import scalar_search_armijo
from scipy.optimize import root
from scipy.optimize import rosen, rosen_der
import pycutest
import time
#need to have the complete ssd code in a .py file to import. 

ModuleNotFoundError: No module named 'SSD_draft'

## Modified Stochastic Subspace Descent for Nonlinear Least Squares

### Gauss-Newton

## Modified G-N Stochastic Subspace Descent for Nonlinear Least Squares
Oracle: ${r_i(x)}^m_{i=1} = {q^\top \nabla r_i(x)}^m_{i=1} = J(x)q$ where q is a directional derivative. 

We know $r(x_k + p) = r(x_k) + J(x_k)p$ is the residual function. The nonlinear least square problem is then $$ \min_x ||r(x_k) + J(x_k)p||^2_2 $$

Apply $p = Pn$ to the SSD algorithm, where P is a projection matrix of size $d \times l$ where $l < d$. After applying the substitution, we obtain the normal equation 
\begin{equation}
-(JP)^\top r = (JP)^\top (JP) n
\end{equation}
solving for $n^*$ we have 
$$ Pn = p = P(P^\top J^\top J P)^{-1} P^\top J^\top r $$
Let $P = Q \cdot S$, where the column space of P and Q is the same and $S^{-1}$ exists. Then 
$$ p = Q(Q^\top J^\top J Q)^{-1} Q^\top J^\top r $$
Let $\tilde{J} = J \cdot Q$, then 
$$ p = Q ( \tilde{J}^\top \tilde{J} ) ^{-1} \tilde{J}^\top r$$
and the algorithm costs $l$ for $l$ directional derivatives. 

For computing purposes, compute by finding
$$ p = \text{arg} \min_{n \in \mathbb{R}} ||\tilde{J}n - r_x||^2_2$$

### Linesearch Algo
Similar to the original SSD version except $\phi'(0) = J p_k$, which is also already computed in current step. No extra step needed. Using armijo condition. 

In [10]:
def gn_ssd(x, cons, l=None, ls=None, max_iter=200, seed=0, amin=1e-10):
    '''
    input: x is the starting point; P is dxl; r the residual function
    Q has Haar distribution
    Assume alpha = 1
    iteration: x = P
    '''

    dim = len(x)
    rng = np.random.default_rng(seed)

    #return variables
    x_history = []
    fx_history = []
    stepsize_history = []

    #subspace dimension
    dim = len(x)
    if l is None:
        # default choice for subspace:
        l = min([max([5, int(np.sqrt(dim)) ]), dim])
    if l > dim: raise ValueError('subspace dim should be < dimension')
    if l == dim: print(' Are you sure you want l == dim?')

    #define f as a function than a value
    def fobj(z): rz = cons(z); return 0.5*np.dot(rz, rz)
    alpha0 = 1.0
    
    for n in range(max_iter):
        r , J = cons(x, gradient=True) #residual and Jacobian
        fval = 0.5 * np.dot(r, r)
        
        Z = rng.standard_normal((dim, l))
        Q, _ = linalg.qr(Z, mode='economic') #economic mode for finding the QR so it's not a full square matrix

        #solving for the step
        J_tild = J @ Q
        pk = np.linalg.lstsq(J_tild, - r, rcond=None)[0]

        if ls == 'on':
            derphi0 = (J_tild.T @ r) @ pk
            direc = Q @ pk
            alpha_ls, nfev, phi1 = linesearch(fobj, x, direc, derphi0, old_fval=fval,alpha0=max(alpha0, amin), amin=amin)
            #if it fails
            if alpha_ls is None: 
                alpha_ls, phi1 = 0.0, fval
            else:
                alpha0 = 2.5 * alpha_ls
            stepsize_history.append(alpha_ls)
            fval = phi1
            x = x + alpha_ls * direc
        elif ls == 'off': #alpha = 1
            x = x + Q @ pk
            fval = float(fval)
        else:
            raise ValueError("please choose linesearch mode.")

        x_history.append(np.array(x))
        fx_history.append(fval)

    return np.array(fx_history), np.array(x_history), np.array(stepsize_history)

In [1]:
nls_prob = pycutest.find_problems(objective='none', constraints='linear quadratic other', regular=True)
nls_prob[:10]

NameError: name 'pycutest' is not defined

In [ ]:
#test the gn_ssd algorithm
print(pycutest.problem_properties('CHANDHEQ'))
p = pycutest.import_problem('CHANDHEQ')
x = p.x0

fx_ls, xs_ls, ls_hist = gn_ssd(x, p.cons, ls = 'on')
# %lprun -f ssd fx, xs, linesearch_hist = ssd(p.obj, p.grad, x, max_iter = int(1e4) )
print(f'size of x_i: {x.size}')
print("with linesearch")
print(fx_ls[0], '→', fx_ls[-1])

#without linesearch
fx, xs, _ = gn_ssd(x, p.cons, ls = 'off')
print("without linesearch")
print(fx[0], '→', fx[-1])
print(p.report())

fig, ax = plt.subplots()
ax.semilogy(fx_ls, label = 'with linesearch')
ax.semilogy(fx, label = 'without linesearch')
ax.legend()
plt.show()

### Algorithm to compare GN-SSD to
* IBCDFO: https://github.com/POptUS/IBCDFO.git
* Cartis' A Randomised Subspace Gauss-Newton Method for Nonlinear Least-Squares (no code tho... would be great comparison)
* DFBGN: https://github.com/numericalalgorithmsgroup/dfbgn (also from the other Cartis paper)

In [ ]:
import ibcdfo

p = pycutest.import_problem('ARGLALE', sifParams={'N': 50, 'M': 100})
n, m = p.n, p.m
X0 = np.asarray(p.x0, dtype=float)
Ffun = p.cons                     # bound method: pinned to THIS p, no late lookup
nf_max = 100 * (n + 1)
Low, Upp = -np.inf*np.ones(n), np.inf*np.ones(n)

# X, F, hF, flag, xk_best = pounders(Ffun, X0, n, nf_max, 1e-13, 0.1, m, Low, Upp)
ibcdfo.run_pounders(Ffun, X0, n, nf_max, 1e-10, 0.1, m, Low, Upp)
print(flag, hF.shape)
print(hF[0], np.dot(Ffun(X0), Ffun(X0)))
f_curve_p = np.minimum.accumulate(hF / 2.0)
print(f_curve_p[0], f_curve_p[-1])  

### Levenberg-Marquardt
L-M is a combination between Gauss-Newton and gradient descent. The problem is (after evaluating Lagrange Multiplier):
$$ \min_p ||Jp + r||^2 + \frac{\lambda}{2}||x_0|| $$
The normal equation for L-M is 
$$ -(JP)^\top r = ((JP)^\top (JP)-\lambda I) n $$
Input: 
* initial $\lambda_0$
* initial $x_0$ size $n \times 1$
* r

Importantly, need to update $\lambda$ each step. I use $D = I$ because we don't compute the whole Jacobian. 

Figure out how to update $\lambda$. Derive the update equation for this. 

7/2/2026 Did not finish this, found a Cartis paper that already does random newton in trust region. No need to follow up. 

In [8]:
def lm_ssd(x, l, r, lam, max_iter=200):
    '''
    for ...:
        evaluate x
        evaluate rho, the actual reduction
        if rho < :
        else: 
        
    '''

    for n in range(max_iter):
        rx = r(x)
        
        Z = np.random.default_rng().standard_normal((dim, l))
        Q, R = linalg.qr(Z, mode='economic') #economic mode for finding the QR so it's not a full square matrix

        #use jax.jvp for jacobian-vector multiplication. Need x and v in tuple. Returns r(x) and Jx
        #use jax.vmap to map it to every row
        J_tild = jax.vmap(lambda v: jax.jvp(r, (x,), (v,))[1])(Q.T)
        p = jnp.linalg.lstsq(J_tild.T @ J_tild - lam @ np.eye(l), - J_tild.T @ rx)[0]

        #compute gain ratio

## Linesearch For SSD
7/6/2026  
Let $$\phi(t) = f(x_k + t p_k) $$ where $\phi(t): \mathbb{R} \rightarrow \mathbb{R} $  
Let $\Phi(t) \approx \phi(t)$, where $\Phi(t)$ is a quadratic/cubic interpolation of points $\phi(0)$, $\phi(t_1)$, and $\phi'(0)$ (and $\phi(t_2)$ if cubic).   
For SSD, $\phi'(0) = - || p_k||^2 $, so I will modify the scipy code where there's no need to calculate derphi. 

In [ ]:
#input alpha and l, independent variables
#0 < alpha < 2l/(d*lambda). Why?

def ssd(x, l, f, dim, alpha=None, max_iter=200):
    '''
    followed algorithm 2.1:
    1: Inputs: α, l. step size, subspace rank
    2: Initialize: x0 . arbitrary initialization
    3: for k = 1, 2, . . . do
    4: Generate P_k
    5: Apply recursion
    '''

    fx = []
    x_history = [np.array(x)]
    gradientFcn = jax.jit(jax.grad(f))
    for n in range(max_iter):
        #generate P using haar distribution (not necessary)
        Z = np.random.default_rng().standard_normal((dim, l)) # 
        Q, R = linalg.qr(Z, mode='economic') #economic mode for finding the QR so it's not a full square matrix
        #lambda step is optional, only required for Haar. 
        # Lambda = np.diag(np.sign(np.diag(R))) # R_ii > 0, ensure unique QR fact. 
        P = np.sqrt(dim / l) * Q 

        #recurse
        grad = gradientFcn(x)
        y = P.T @ grad
        step = P @ y
        if alpha is None:
            p_k = - step
            result = line_search(
                    lambda x: float(f(jnp.array(x))),
                    lambda x: np.array(gradientFcn(jnp.array(x))),
                    np.array(x),
                    p_k,
                )
            alpha = result[0]
            x = x + alpha * p_k
        else:
            x = x - alpha* step
        
        #plug x into function
        fx.append(float(f(jnp.array(x))))
        x_history.append(np.array(x))

    return fx, x_history